# Activity 6 – Appendix: Additional Exercises
**Student ID:** `TUPM-26-3607`

| Parameter | Rule | Value |
|---|---|---|
| `SEED_DIGIT` | last character of the Student ID | **7** |
| `ID_SUM` | sum of all digits in the Student ID | **24** |

| Exercise | Derived parameter | Value |
|---|---|---|
| 1. Web Server Access Log | Records processed = 1000 + ID_SUM | **1024** |
| 1. Web Server Access Log | Target status class = (SEED_DIGIT % 4) + 2 | **5** → 5xx |
| 2. Mobile Device Usage | Operating system (SEED_DIGIT odd) | **iOS** |
| 2. Mobile Device Usage | Behavior class = (SEED_DIGIT % 5) + 1 | **3** |
| 3. Cybersecurity Incidents | Target year = 2015 + SEED_DIGIT | **2022** |

Only Python built-ins and standard-library modules are used (no Pandas / NumPy).
Place the three Kaggle CSV files inside the `data/` folder (it is git-ignored, so the raw datasets are never uploaded):

- `data/weblog.csv`
- `data/user_behavior_dataset.csv`
- `data/Global_Cybersecurity_Threats_2015-2024.csv`

## Setup: Student Configuration and Workspace

In [ ]:
# [CELL 1] STUDENT CONFIGURATION AND WORKSPACE
from pathlib import Path
from collections import Counter
import csv
import json
import shutil

STUDENT_ID = "TUPM-26-3607"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Student ID:", STUDENT_ID)
print("Seed digit:", SEED_DIGIT)
print("ID sum:", ID_SUM)
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

In [ ]:
# [CELL 2] SHARED HELPER FUNCTIONS
def require_file(path):
    # Stop early with a clear message if a dataset is missing.
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}. Download it from Kaggle and place it in the data/ folder."
        )


def clean_header(reader):
    # Remove stray spaces / BOM characters from the header names.
    reader.fieldnames = [name.strip().lstrip("\ufeff") for name in reader.fieldnames]


def make_backup(source_path):
    # Copy a file into the backup folder and verify the copy exists and matches in size.
    backup_path = BACKUP_DIR / source_path.name
    shutil.copy2(source_path, backup_path)
    verified = backup_path.exists() and backup_path.stat().st_size == source_path.stat().st_size
    return backup_path, verified

---
# Exercise 1: Web Server Access Log Analysis
**Mission:** process a student-specific number of web requests, identify one HTTP response class, and archive the findings.

In [ ]:
# [CELL 3] EXERCISE 1 - DATA GENERATION RULES
WEBLOG_PATH = DATA_DIR / "weblog.csv"

RECORDS_TO_PROCESS = 1000 + ID_SUM               # first 1000 + ID_SUM data records
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2       # 2 -> 2xx, 3 -> 3xx, 4 -> 4xx, 5 -> 5xx
STATUS_LOW = TARGET_STATUS_CLASS * 100
STATUS_HIGH = STATUS_LOW + 99

print("Records to process:", RECORDS_TO_PROCESS)
print("Target status class:", f"{TARGET_STATUS_CLASS}xx ({STATUS_LOW}-{STATUS_HIGH})")
print("Dataset exists:", WEBLOG_PATH.exists())

In [ ]:
# [CELL 4] EXERCISE 1 - READ LINE BY LINE, EXTRACT FIELDS, SELECT TARGET CLASS
def parse_status(raw):
    # Return the HTTP status as an int, or None if it is missing / invalid.
    try:
        return int(str(raw).strip())
    except (ValueError, TypeError):
        return None


def extract_path(request):
    # "GET /index.html HTTP/1.1" -> "/index.html"
    parts = request.split()
    return parts[1] if len(parts) >= 2 else request


require_file(WEBLOG_PATH)

records_processed = 0
invalid_records = 0
status_counts = Counter()
url_counts = Counter()
selected_records = []

with open(WEBLOG_PATH, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)           # reads one row at a time (no full load)
    clean_header(reader)

    for row in reader:
        if records_processed >= RECORDS_TO_PROCESS:
            break
        records_processed += 1

        ip = (row.get("IP") or "").strip()
        timestamp = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status = parse_status(row.get("Staus"))      # the dataset column is spelled "Staus"

        if not ip or not timestamp or not url or status is None:
            invalid_records += 1
            continue

        status_counts[status] += 1

        if STATUS_LOW <= status <= STATUS_HIGH:
            selected_records.append(
                {"IP": ip, "Time": timestamp, "URL": url, "Status": status}
            )
            url_counts[extract_path(url)] += 1

top_five_urls = sorted(url_counts.items(), key=lambda item: -item[1])[:5]   # stable: ties keep first-seen order

print("Records processed:", records_processed)
print("Invalid / missing records skipped:", invalid_records)
print("Records in target class:", len(selected_records))
print("Count per status code:", dict(sorted(status_counts.items())))
print("Five most requested URLs:")
for rank, (url, count) in enumerate(top_five_urls, 1):
    print(f"  {rank}. {url} ({count})")
if not top_five_urls:
    print("  (no requests fall inside the target status class)")

In [ ]:
# [CELL 5] EXERCISE 1 - EXPORT FILTERED CSV, WRITE JSON SUMMARY, BACKUP AND VERIFY
filtered_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_weblog_filtered_{TARGET_STATUS_CLASS}xx.csv"
summary_json_path = OUTPUT_DIR / f"{STUDENT_ID}_weblog_summary.json"

with open(filtered_csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", "Status"])
    writer.writeheader()
    writer.writerows(selected_records)

weblog_summary = {
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_processed,
    "invalid_records": invalid_records,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": len(selected_records),
    "count_per_status_code": {str(code): n for code, n in sorted(status_counts.items())},
    "top_five_urls": [{"url": url, "count": count} for url, count in top_five_urls],
    "note": "No records in target class" if not selected_records else "OK",
}

with open(summary_json_path, "w", encoding="utf-8") as file:
    json.dump(weblog_summary, file, indent=4)

weblog_backup_path, weblog_backup_ok = make_backup(summary_json_path)

print("Filtered CSV:", filtered_csv_path)
print("JSON summary:", summary_json_path)
print("Backup:", weblog_backup_path)
print("Backup exists:", weblog_backup_ok)

In [ ]:
# [CELL 6] EXERCISE 1 - FINAL OUTPUT (ASSESSMENT DATA)
print("=" * 60)
print("EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS")
print("=" * 60)
print("Records processed      :", records_processed)
print("Target status class    :", f"{TARGET_STATUS_CLASS}xx")
print("Records in target class:", len(selected_records))
print("Count per status code  :", dict(sorted(status_counts.items())))
print("Five most requested URLs:")
for rank, (url, count) in enumerate(top_five_urls, 1):
    print(f"   {rank}. {url} -> {count}")
print("Filtered CSV file      :", filtered_csv_path.name)
print("JSON summary file      :", summary_json_path.name)
print("Backup verification    :", "VERIFIED" if weblog_backup_ok else "FAILED", "-", weblog_backup_path.name)

---
# Exercise 2: Mobile Device Usage Profiling
**Mission:** filter device records using student-specific criteria and export a verified usage summary.

In [ ]:
# [CELL 7] EXERCISE 2 - DATA GENERATION RULES
MOBILE_PATH = DATA_DIR / "user_behavior_dataset.csv"

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

EXPORT_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

print("Target operating system:", TARGET_OS)
print("Target behavior class  :", TARGET_BEHAVIOR_CLASS)
print("Dataset exists:", MOBILE_PATH.exists())

In [ ]:
# [CELL 8] EXERCISE 2 - READ, FILTER BY OS AND BEHAVIOR CLASS, EXPORT WHILE STREAMING
def to_float(raw):
    try:
        return float(str(raw).strip())
    except (ValueError, TypeError):
        return None


require_file(MOBILE_PATH)

mobile_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_mobile_{TARGET_OS}_class{TARGET_BEHAVIOR_CLASS}.csv"

total_records = 0
os_matches = 0
selected_count = 0
invalid_records = 0
battery_total = 0.0
data_total = 0.0
battery_n = 0
data_n = 0

with open(MOBILE_PATH, "r", encoding="utf-8", newline="") as source, \
     open(mobile_csv_path, "w", encoding="utf-8", newline="") as target:

    reader = csv.DictReader(source)
    clean_header(reader)
    writer = csv.DictWriter(target, fieldnames=EXPORT_FIELDS)
    writer.writeheader()

    for row in reader:
        total_records += 1

        os_name = (row.get("Operating System") or "").strip()
        if os_name.lower() != TARGET_OS.lower():
            continue
        os_matches += 1

        try:
            behavior_class = int(str(row.get("User Behavior Class")).strip())
        except (ValueError, TypeError):
            invalid_records += 1
            continue
        if behavior_class != TARGET_BEHAVIOR_CLASS:
            continue

        selected_count += 1

        battery = to_float(row.get("Battery Drain (mAh/day)"))
        data_usage = to_float(row.get("Data Usage (MB/day)"))
        if battery is not None:
            battery_total += battery
            battery_n += 1
        if data_usage is not None:
            data_total += data_usage
            data_n += 1

        writer.writerow({field: (row.get(field) or "").strip() for field in EXPORT_FIELDS})

average_battery = battery_total / battery_n if battery_n else None
average_data = data_total / data_n if data_n else None

print("Total records in dataset:", total_records)
print("Records matching OS:", os_matches)
print("Records in target behavior class:", selected_count)
print("Invalid class values skipped:", invalid_records)
print("Average battery drain (mAh/day):", round(average_battery, 2) if average_battery is not None else "N/A - no matching records")
print("Average data usage (MB/day):", round(average_data, 2) if average_data is not None else "N/A - no matching records")
print("Exported CSV:", mobile_csv_path)

In [ ]:
# [CELL 9] EXERCISE 2 - JSON SUMMARY, COUNT VERIFICATION, BACKUP
# Re-read the exported CSV and count its data rows
with open(mobile_csv_path, "r", encoding="utf-8", newline="") as file:
    exported_rows = sum(1 for _ in csv.DictReader(file))

count_verified = exported_rows == selected_count

mobile_summary_path = OUTPUT_DIR / f"{STUDENT_ID}_mobile_summary.json"
mobile_summary = {
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": selected_count,
    "average_battery_drain_mAh_day": round(average_battery, 2) if average_battery is not None else None,
    "average_data_usage_MB_day": round(average_data, 2) if average_data is not None else None,
    "exported_rows": exported_rows,
    "count_verified": count_verified,
    "note": "No matching records" if selected_count == 0 else "OK",
}

with open(mobile_summary_path, "w", encoding="utf-8") as file:
    json.dump(mobile_summary, file, indent=4)

mobile_backup_path, mobile_backup_ok = make_backup(mobile_csv_path)
mobile_backup_size = mobile_backup_path.stat().st_size

print("Exported rows:", exported_rows, "| Selected records:", selected_count)
print("Count verification:", "MATCH" if count_verified else "MISMATCH")
print("JSON summary:", mobile_summary_path)
print("Backup:", mobile_backup_path)
print("Backup exists:", mobile_backup_ok)
print("Backup size:", mobile_backup_size, "bytes")

In [ ]:
# [CELL 10] EXERCISE 2 - FINAL OUTPUT (ASSESSMENT DATA)
print("=" * 60)
print("EXERCISE 2: MOBILE DEVICE USAGE PROFILING")
print("=" * 60)
print("Operating system / class     :", TARGET_OS, "/ class", TARGET_BEHAVIOR_CLASS)
print("Total records in dataset     :", total_records)
print("Records matching OS          :", os_matches)
print("Records in target class      :", selected_count)
print("Average battery drain (mAh/d):", round(average_battery, 2) if average_battery is not None else "N/A")
print("Average data usage (MB/day)  :", round(average_data, 2) if average_data is not None else "N/A")
print("Count verification           :", "VERIFIED" if count_verified else "CHECK", f"({exported_rows} exported = {selected_count} selected)")
print("Exported CSV file            :", mobile_csv_path.name)
print("JSON summary file            :", mobile_summary_path.name)
print("Backup file and size         :", mobile_backup_path.name, "-", mobile_backup_size, "bytes")

---
# Exercise 3: Cybersecurity Incident Report
**Mission:** process records for a student-specific year, summarize attack types, and archive the findings.

In [ ]:
# [CELL 11] EXERCISE 3 - DATA GENERATION RULES
CYBER_PATH = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

TARGET_YEAR = 2015 + SEED_DIGIT
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print("Target year:", TARGET_YEAR)
print("Loss field :", LOSS_FIELD)
print("Count field:", COUNT_FIELD)
print("Dataset exists:", CYBER_PATH.exists())

In [ ]:
# [CELL 12] EXERCISE 3 - READ CSV AND SELECT INCIDENTS IN THE TARGET YEAR
require_file(CYBER_PATH)

cyber_total_records = 0
invalid_year_records = 0
year_incidents = []          # incidents in the target year, in original dataset order
cyber_fieldnames = []

with open(CYBER_PATH, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    clean_header(reader)
    cyber_fieldnames = list(reader.fieldnames)

    for row in reader:
        cyber_total_records += 1
        try:
            year = int(str(row.get("Year")).strip())
        except (ValueError, TypeError):
            invalid_year_records += 1
            continue
        if year == TARGET_YEAR:
            year_incidents.append(row)

print("Total records in dataset:", cyber_total_records)
print("Records with invalid year skipped:", invalid_year_records)
print("Incidents in target year:", len(year_incidents))

In [ ]:
# [CELL 13] EXERCISE 3 - COUNT ATTACK TYPES, TOTAL / AVERAGE LOSS, FIVE COSTLIEST
attack_counts = Counter()
valid_losses = []            # (original_index, loss, row)
invalid_loss_records = 0

for index, row in enumerate(year_incidents):
    attack_counts[(row.get(COUNT_FIELD) or "").strip() or "Unknown"] += 1

    try:
        loss = float(str(row.get(LOSS_FIELD)).strip())
        if loss != loss or loss < 0:        # reject NaN and negative values
            raise ValueError
    except (ValueError, TypeError):
        invalid_loss_records += 1
        continue
    valid_losses.append((index, loss, row))

total_loss = sum(loss for _, loss, _ in valid_losses)
average_loss = total_loss / len(valid_losses) if valid_losses else None

# Descending loss; sorted() is stable so ties keep the original dataset order
ranked = sorted(valid_losses, key=lambda item: -item[1])
five_costliest = ranked[:5]

print("Count per attack type:", dict(attack_counts))
print("Invalid / missing loss values skipped:", invalid_loss_records)
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2) if average_loss is not None else "N/A - no valid incidents")
print("Five costliest incidents:")
for rank, (_, loss, row) in enumerate(five_costliest, 1):
    print(f"  {rank}. {row.get('Country')} | {row.get('Attack Type')} | {row.get('Target Industry')} | ${loss} M")
if not five_costliest:
    print("  (no valid incidents in the target year)")

In [ ]:
# [CELL 14] EXERCISE 3 - EXPORT FILTERED CSV, TOP-FIVE JSON, SUMMARY JSON, VERIFIED BACKUP
cyber_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_{TARGET_YEAR}_filtered.csv"
cyber_top5_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_{TARGET_YEAR}_top5.json"
cyber_summary_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_{TARGET_YEAR}_summary.json"

with open(cyber_csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=cyber_fieldnames)
    writer.writeheader()
    writer.writerows(year_incidents)

top_five_records = [dict(row, **{"Rank": rank}) for rank, (_, loss, row) in enumerate(five_costliest, 1)]
with open(cyber_top5_path, "w", encoding="utf-8") as file:
    json.dump(top_five_records, file, indent=4)

cyber_summary = {
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_year": TARGET_YEAR,
    "total_records": cyber_total_records,
    "incidents_in_target_year": len(year_incidents),
    "count_per_attack_type": dict(attack_counts),
    "invalid_loss_records": invalid_loss_records,
    "total_financial_loss_usd_millions": round(total_loss, 2),
    "average_financial_loss_usd_millions": round(average_loss, 2) if average_loss is not None else None,
    "five_costliest_file": cyber_top5_path.name,
    "note": "No incidents in target year" if not year_incidents else "OK",
}
with open(cyber_summary_path, "w", encoding="utf-8") as file:
    json.dump(cyber_summary, file, indent=4)

cyber_backup_path, cyber_backup_ok = make_backup(cyber_summary_path)

print("Filtered CSV:", cyber_csv_path)
print("Top-five JSON:", cyber_top5_path)
print("JSON summary:", cyber_summary_path)
print("Backup:", cyber_backup_path)
print("Backup exists and size matches:", cyber_backup_ok)

In [ ]:
# [CELL 15] EXERCISE 3 - TEXT REPORT
cyber_report_path = OUTPUT_DIR / f"{STUDENT_ID}_cyber_{TARGET_YEAR}_report.txt"

with open(cyber_report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-" * 50 + "\n")
    report.write(f"Student ID: {STUDENT_ID}\n")
    report.write(f"Target Year: {TARGET_YEAR}\n")
    report.write(f"Total Records in Dataset: {cyber_total_records}\n")
    report.write(f"Incidents in Target Year: {len(year_incidents)}\n\n")

    report.write("Incident Count per Attack Type\n")
    if attack_counts:
        for attack, count in attack_counts.most_common():
            report.write(f"  {attack}: {count}\n")
    else:
        report.write("  (none)\n")

    report.write("\nFinancial Loss (USD millions)\n")
    report.write(f"  Total: {total_loss:.2f}\n")
    if average_loss is not None:
        report.write(f"  Average: {average_loss:.2f}\n")
    else:
        report.write("  Average: N/A\n")
    report.write(f"  Invalid loss values skipped: {invalid_loss_records}\n")

    report.write("\nFive Costliest Incidents\n")
    if five_costliest:
        for rank, (_, loss, row) in enumerate(five_costliest, 1):
            report.write(
                f"  {rank}. {row.get('Country')} | {row.get('Attack Type')} | "
                f"{row.get('Target Industry')} | ${loss:.2f} M\n"
            )
    else:
        report.write("  (none)\n")

    report.write("\nOutput Files\n")
    report.write(f"  Filtered CSV: {cyber_csv_path.name}\n")
    report.write(f"  Top-five JSON: {cyber_top5_path.name}\n")
    report.write(f"  JSON summary: {cyber_summary_path.name}\n")
    report.write(f"  Backup: {cyber_backup_path.name} ({'verified' if cyber_backup_ok else 'NOT verified'})\n")

print("Report created:", cyber_report_path)
print("Exists:", cyber_report_path.exists())

In [ ]:
# [CELL 16] EXERCISE 3 - FINAL OUTPUT (ASSESSMENT DATA)
print("=" * 60)
print("EXERCISE 3: CYBERSECURITY INCIDENT REPORT")
print("=" * 60)
print("Target year                   :", TARGET_YEAR)
print("Total records in dataset      :", cyber_total_records)
print("Incidents in target year      :", len(year_incidents))
print("Count per attack type         :", dict(attack_counts))
print("Total financial loss (USD M)  :", round(total_loss, 2))
print("Average financial loss (USD M):", round(average_loss, 2) if average_loss is not None else "N/A")
print("Five costliest incidents      :")
for rank, (_, loss, row) in enumerate(five_costliest, 1):
    print(f"   {rank}. {row.get('Country')} | {row.get('Attack Type')} | ${loss} M")
print("Filtered CSV file             :", cyber_csv_path.name)
print("JSON summary file             :", cyber_summary_path.name)
print("Backup verification           :", "VERIFIED" if cyber_backup_ok else "FAILED", "-", cyber_backup_path.name)
print("Report file                   :", cyber_report_path.name)

---
# Deployment (run in the VS Code terminal, not in a notebook cell)

Keep this notebook inside `Lab6_File_Handling_Additional_Exercises/` in your local repository.

```bash
mkdir Lab6_File_Handling_Additional_Exercises
git checkout -b activity6-additional-exercises
# move this notebook (and any .py files) into Lab6_File_Handling_Additional_Exercises/
git status
git add Lab6_File_Handling_Additional_Exercises
git commit -m "Activity 6 Additional Exercises: File Handling"
git push -u origin activity6-additional-exercises
```

`data/` is already in `.gitignore`, so the large raw datasets are not uploaded. After pushing, open the branch on GitHub and confirm the notebook renders with its outputs.